In [7]:
!pip install pennylane
import pandas as pd 
from qiskit import QuantumCircuit
import matplotlib.pyplot as plt
import pennylane as mod
import numpy as np
from PIL import Image
import os

In [8]:
# Constants/parameters

# Number of Qubits we will use, depends on how many photos we will use and stuff
n_qubits = 8

# Number of layers we will use to train the model
n_layers = 3

# Patches: Images will be broken into 4x4 patches; Each image will have 16patches for all 256 pixels
#keep in mind if you want to do larger/smaller images you will have to update the patch sizes
n_patch_size = 4 
n_patches = 16

#random seed set to 42, pls dont change i'll cry
np.random.seed(42)

device = mod.device("default.qubit", wires = n_qubits)

In [9]:
# Variational Circuit
#hi mi llama Abbie

@mod.qnode(device)
def variational_Circuit(inputs, weights):

    #weights for the model, 
    weights = weights.reshape(n_layers, n_qubits, 3)

    #encode patches into quantum state
    for i in range(n_qubits):
        mod.RY(inputs[i], wires=i)
    
    #variational layers
    for layer in range(n_layers):
        #rotation layer
        for qubit in range(n_qubits):
            mod.RX(weights[layer, qubit, 0], wires=qubit)
            mod.RY(weights[layer, qubit, 1], wires=qubit)
            mod.RZ(weights[layer, qubit, 2], wires=qubit)
        
        # Entanglement layer - connect all qubits in a ring
        for qubit in range(n_qubits):
            mod.CNOT(wires=[qubit, (qubit + 1) % n_qubits])
            
    # Return expectation values from each qubit
    return [mod.expval(mod.PauliZ(wires=i)) for i in range(n_qubits)]


In [10]:
#HELPER FUNCTIONS

def generate_patches(image, size):
    # splits image into patches of size size x size

    # Convert flat 1D image to 2D if needed
    if image.ndim == 1:
        img_dim = int(np.sqrt(len(image)))
        image = image.reshape((img_dim, img_dim))
        
    height = len(image)
    width = len(image[0])
    patches = []
    for i in range(0, height - size + 1, size):
        for j in range(0, width - size + 1, size):
            patch = image[i:i + size, j:j + size].flatten()
            patches.append(patch)
    return patches

#ADDED PREPARE_QUANTUM_DATA FUNC
def prepare_quantum_data(image):
    patches = generate_patches(image, n_patch_size)
    # normalize to [0, π]
    return [patch / np.max(np.abs(patch)) * np.pi for patch in patches]

def add_gaussian_noise(patch, noise_level):
    #.normal follows a gaussian distribution to generate random samples
    #in our case we use it for noise
    #CHANGED size=image.shape to size=patch.shape
    noise = np.random.normal(0, noise_level, size=patch.shape)

    #combine the noise to the patch and return
    noisy_patch = patch + noise
    return noisy_patch

def cost_function(weights, predicted_patches, clean_patches):
    #define cost/weighting function for training

    loss = 0
    
    for noisy_patch, clean_patch in zip(predicted_patches, clean_patches):
        #rescale input to fit within [-π, π] for quantum encoding
        quantum_inputs = noisy_patch[:n_qubits]
        
        #get circuit output
        #CHANGED TO VAR CIRCUIT FUNC
        circuit_output = variational_Circuit(quantum_inputs, weights)
        
        #rescale clean patch to compare with the noisy image
        target_output = clean_patch[:n_qubits]
        #ADDED SMALL NUMBER IN DENOMINATOR TO AVOID /0
        target_output_normalized = (target_output / (np.max(np.abs(target_output)) + 1e-8)) * 2 - 1
        
        #calculate MSE
        #add together the loss value from each patch 
        patch_loss = np.mean((np.array(circuit_output) - target_output_normalized)**2)
        loss += patch_loss
    
    return loss / len(predicted_patches)

def denoise_with_quantum_circuit(noisy_image, weights):

    #prepare patches
    noisy_patches = prepare_quantum_data(noisy_image)
    denoised_patches = []
    
    for patch in noisy_patches:
        #use quantum circuit to denoise
        quantum_inputs = patch[:n_qubits]
        #CHANGED TO VAR CIRCUIT FUNC
        circuit_output = np.array(variational_Circuit(quantum_inputs, weights))
        
        #rescale output from [-1, 1] to original scale
        denoised_patch = (circuit_output + 1) / 2

        #ADDED APPEND DENOISED PATCH TO PATCHES LIST
        denoised_patches.append(denoised_patch)
      
    #reconstruct image from patches
    img_dim = int(np.sqrt(len(noisy_image)))
    denoised_image = np.zeros((img_dim, img_dim))
    
    patch_idx = 0
    for i in range(0, img_dim, n_patch_size):
        for j in range(0, img_dim, n_patch_size):
            patch = denoised_patches[patch_idx].reshape(n_patch_size, n_patch_size)
            denoised_image[i:i+n_patch_size, j:j+n_patch_size] = patch
            patch_idx += 1
    
    return denoised_image.flatten()


def load_images(image_dir):
    #get all image files from directory
    #update valid extensions if you add images with different file types
    image_files = []
    valid_extensions = {'.png'}
    
    for file in os.listdir(image_dir):
        ext = os.path.splitext(file)[1].lower()
        if ext in valid_extensions:
            image_files.append(os.path.join(image_dir, file))
    
    #sort files to ensure consistent loading order
    image_files.sort()
    
    #load images
    images = []
    for img_path in image_files:
        #open and convert to grayscale
        #our images are already grayscale but this makes training easier for future images
        with Image.open(img_path) as img:
            img = img.convert('L')
            
            #verify size is 16x16
            #you can void or change this for future images if youre using different sizes
            if img.size != (16, 16):
                print(f"Image {img_path} is not 16x16, resizing")
                img = img.resize((16, 16), Image.LANCZOS)
            
            #convert to numpy array and normalize to [0,1]
            img_array = np.array(img).astype(np.float32) / 255.0
            
            #flatten to 1D array
            images.append(img_array.flatten())
    
    return np.array(images)

In [11]:
#TRAINING LOOP

def train_quantum_diffusion_model(clean_images, n_epochs=100, batch_size=4):


    #ADDED INITIALIZE SCHED
    noise_schedule = [0.05, 0.1, 0.2]

    # #add weights
    # weights = initial_weights

    #CHANGED WEIGHT INITIALIZATION
    weights = mod.numpy.array(mod.numpy.random.randn(n_layers * n_qubits * 3), requires_grad=True)

    #optimizer function from pennylane
    opt = mod.AdamOptimizer(stepsize=0.01)
    
    losses = []
    
    for epoch in range(n_epochs):
        batch_indices = np.random.choice(len(clean_images), size=batch_size, replace=False)
        batch_images = [clean_images[i] for i in batch_indices]
        
        #for each noise level in our schedule
        for noise_level in noise_schedule:
            #create clean and noisy patches
            all_clean_patches = []
            all_noisy_patches = []
            
            for image in batch_images:
                #add noise according to current level
                #CHANGED TO CORRENT FUNCTION NAME
                noisy_image = add_gaussian_noise(image, noise_level)
                
                #convert the images into patches
                clean_patches = prepare_quantum_data(image)
                noisy_patches = prepare_quantum_data(noisy_image)
                
                all_clean_patches.extend(clean_patches)
                all_noisy_patches.extend(noisy_patches)
            
            #update the weights
            weights = opt.step(lambda w: cost_function(w, all_noisy_patches, all_clean_patches), weights)
            
            # Calculate loss for monitoring
            loss = cost_function(weights, all_noisy_patches, all_clean_patches)
            losses.append(loss)
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch}: Loss = {loss:.4f}")
    
    return weights, losses

In [12]:
#THE MAIN

#load images
img_dir = "/home/jovyan/Quantum_Diffusion_Model/Number_Images"
clean_images = load_images(img_dir)

#train the model
trained_weights, training_losses = train_quantum_diffusion_model(clean_images)

# test on a noisy image
test_image = clean_images[0]
#CHANGED TO CORRECT FUNCTION NAME
noisy_test_image = add_gaussian_noise(test_image, 0.1)
denoised_image = denoise_with_quantum_circuit(noisy_test_image, trained_weights)

# plot results
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.title("Original")
plt.imshow(test_image.reshape(int(np.sqrt(len(test_image))), int(np.sqrt(len(test_image)))), cmap='gray')
plt.subplot(1, 3, 2)
plt.title("Noisy")
plt.imshow(noisy_test_image.reshape(int(np.sqrt(len(test_image))), int(np.sqrt(len(test_image)))), cmap='gray')
plt.subplot(1, 3, 3)
plt.title("Denoised")
plt.imshow(denoised_image.reshape(int(np.sqrt(len(test_image))), int(np.sqrt(len(test_image)))), cmap='gray')
plt.show()


ValueError: setting an array element with a sequence.